**1.Setup**

In [10]:
%pip install -q numpy pandas matplotlib seaborn scipy

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

np.random.seed(42)
pd.set_option("display.max_columns",None)
pd.set_option("display.float_format","{:.3f}".format)
sns.set_style("whitegrid")

**2.Load the Dataset**

In [15]:
df = pd.read_csv("insurance.csv")

In [16]:
df.shape

(1338, 7)

In [17]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


## Column Description and Feature Type

| Column       | Description                        | Feature Type                |
| ------------ | ---------------------------------- | --------------------------- |
| **age**      | Age of the individual              | Numerical(continuous)       |
| **sex**      | Gender of the customer             | Categorical                 |
| **bmi**      | Body Mass Index of the customer    | Numerical(continuous)       |
| **children** | Number of children/dependents      | Numerical(discrete)         |
| **smoker**   | Whether the customer is a smoker   | Categorical(binary)         |
| **region**   | Residential region of the customer | Categorical                 |
| **charges**  | Medical insurance charges          | Numerical (continuous,targe)|


### 3.Data Quality Checks

**3.1 Basic Dataset Overview**

In [19]:
# identify numerical & categorical features
num_cols = ["age","bmi","children","charges"]
cat_cols = ["sex","smoker","region"]

print("\nNumerical columns:",num_cols)
print("Categorical columns:",cat_cols)


Numerical columns: ['age', 'bmi', 'children', 'charges']
Categorical columns: ['sex', 'smoker', 'region']


In [24]:
print("Shape:",df.shape)
print("Column names:\n",df.columns.tolist())

print("-"*50)
print("\nData types:")
print(df.dtypes)
print("-"*50)
print("\nUnique values per column:")
print(df.nunique())

Shape: (1338, 7)
Column names:
 ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
--------------------------------------------------

Data types:
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object
--------------------------------------------------

Unique values per column:
age           47
sex            2
bmi          548
children       6
smoker         2
region         4
charges     1337
dtype: int64


In [26]:
print("\nFirst pass info:")
print(df.info())


First pass info:
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB
None


In [27]:
# descriptive statistics
df[num_cols].describe()

,age,bmi,children,charges
count,1338.000,1338.000,1338.000,1338.000
mean,39.207,30.663,1.095,13270.422
std,14.050,6.098,1.205,12110.011
min,18.000,15.960,0.000,1121.874
25%,27.000,26.296,0.000,4740.287
50%,39.000,30.400,1.000,9382.033
75%,51.000,34.694,2.000,16639.913
max,64.000,53.130,5.000,63770.428


### 3.2 Missing Values - Analysis

In [28]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [30]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count":missing_count,
    "missing_percent":missing_percent.round(2)
})
print("Missing values summary:")
missing_summary

Missing values summary:


,missing_count,missing_percent
age,0,0.000
sex,0,0.000
bmi,0,0.000
children,0,0.000
smoker,0,0.000
region,0,0.000
charges,0,0.000


**NOTE:No missing values in the dataset.Still need to verify encoded missingvalues.** 

**example:-999,999,unknown**

In [31]:
for col in df.columns:
    print(df[col].value_counts())
    print("-"*50)

age
18    69
19    68
46    29
52    29
48    29
20    29
45    29
47    29
51    29
50    29
28    28
25    28
23    28
27    28
22    28
26    28
24    28
21    28
53    28
54    28
49    28
31    27
30    27
41    27
40    27
43    27
44    27
29    27
42    27
33    26
32    26
56    26
34    26
55    26
57    26
37    25
59    25
35    25
38    25
36    25
58    25
39    25
60    23
62    23
63    23
61    23
64    22
Name: count, dtype: int64
--------------------------------------------------
sex
male      676
female    662
Name: count, dtype: int64
--------------------------------------------------
bmi
32.300    13
28.310     9
28.880     8
34.100     8
30.800     8
          ..
53.130     1
39.710     1
32.870     1
44.700     1
30.970     1
Name: count, Length: 548, dtype: int64
--------------------------------------------------
children
0    574
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64
--------------------------------------------------
smoker
no 

Note:No encoded missing values

### 3.3. Duplicates

In [37]:
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()

print("Number of duplicate rows:",num_duplicates)
print("Duplicated row:\n",df[duplicate_mask])

# If you want remove duplicates
df_no_duplicates = df.drop_duplicates()
print("Shape after dropping duplicates:",df_no_duplicates.shape)

Number of duplicate rows: 1
Duplicated row:
      age   sex    bmi  children smoker     region  charges
581   19  male 30.590         0     no  northwest 1639.563
Shape after dropping duplicates: (1337, 7)


In [38]:
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')

#### 3.4. Data Type Validation

In [40]:
df["bmi"].dtype

dtype('float64')

In [41]:
expected_types = {
    "age" : "int64",
    "sex" : "object",
    "bmi" : "float64",
    "children" : "int64",
    "smoker" : "object",
    "region" : "object",
    "charges" : "float64"
}

print("Data Type Validation:")
for col,expected in expected_types.items():
    if col in df.columns:
        actual = df[col].dtype
        print(f"{col}:actual = {actual},expected={expected}")       

Data Type Validation:
age:actual = int64,expected=int64
sex:actual = str,expected=object
bmi:actual = float64,expected=float64
children:actual = int64,expected=int64
smoker:actual = str,expected=object
region:actual = str,expected=object
charges:actual = float64,expected=float64


### 3.5 Constant and quassi constant columns

In [ ]:
n_rows = len(df)
nunique = df.nunique()
# print(n_rows)
# print("NUNIQUE\n",nunique)
constant_cols = nunique[nunique == 1].index.tolist()
print("Constant columns:",constant_cols)



Constant columns: []
